### Ein einfacher Timer zu Klasse `Game` hinzuf&uuml;gen. 
Falls der Timer läuft, wird ein async-Task alle 0.01 Sekunden aufgerufen und
updated die Anzeige des Timers (die Variable `displayed_time`).  
Die Funktion `time` des Modules `time` liefert die Anzahl Sekunden, die seit dem 1.1.1970, 00:00:00, vergangen sind.  
**Attribute von Timer**:
- `is_running`: True, falls der Timer läuft
- `displayed_time`: float, Wert des Timers in Sekunden
- `t0`: float, gewählt, so dass `displayed_time` gleich `time.time() - t0`.

**Methoden von Timer**:
- `start`
- `stop`
- `reset`

In [ ]:
import time
import asyncio


class Timer:
    def __init__(self):
        self.is_running = False
        self.displayed_time = 0

    def start(self):
        if not self.is_running:
            self.t0 = time.time() - self.displayed_time
            self._run()

    def stop(self):
        self.is_running = False

    def reset(self):
        self.displayed_time = 0

    def _run(self):
        async def tick():
            while self.is_running:
                self.displayed_time = time.time() - self.t0
                await asyncio.sleep(0.01)

        self.is_running = True
        self.task = asyncio.create_task(tick(), name='tick')

    def __repr__(self):
        return f'{'running' if self.is_running else 'stopped'} {self.displayed_time}'

In [ ]:
timer = Timer()

In [ ]:
timer

In [ ]:
timer.start()

In [ ]:
timer.stop()

In [ ]:
timer.reset()

### Timer zu einer Klasse Game hinzufügen
Wir machen den Timer zu einen Observable (siehe `timer.py`).
Das Game-Objekt registriert dann ein Callback, welches die Beobachter des Game-Objekts informiert.

Nachstehend eine View und ein Controller.
Drücken von Enter soll den Timer starten und stoppen, Backspace setzt den Timer wieder auf 0.

In [1]:
from model_view_controller import Observable, notify, BaseView
from ipycanvas import hold_canvas
from timer import Timer


class Game(Observable):
    def __init__(self):
        self.timer = Timer()
        self.timer.register_callback(self.update_timer)

    @notify
    def update_timer(self, event, displayed_time):
        return displayed_time


class View(BaseView):
    def __init__(self, game):
        super().__init__(game)
        self.show_time()

    def show_time(self):
        with hold_canvas(self.canvas):
            self.canvas.clear()
            s = f'{self.game.timer.displayed_time:06.2f}'  # Zeit als String der Form '012.34'
            self.canvas.fill_text(s, 30, 10)

    def update(self, event, data):
        if event.endswith('timer'):
            self.show_time()


game = Game()
view = View(game)
view

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

In [2]:
game.timer.start()

In [3]:
game.timer.stop()

In [4]:
game.timer.reset()

In [5]:
from model_view_controller import Controller


def key_handler(self, key, state):
    timer = self.game.timer
    if key == 'Enter':
        if timer.is_running:
            timer.is_running = False
        else:
            timer.start()

    elif key == 'Backspace':
        timer.reset()


game = Game()
view = View(game)
controller = Controller(game, view, callbacks=(), key_handler=key_handler)
controller

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

### Aufgabe
Fügen einen Timer zu `shooter.py` hinzu und speichere das File als `shooter_mit_timer.py`. 